# 🪝 Notebook 8: Webhooks

So far we've focused on the **client → server** hop (polling, SSE, WebSockets).
**Webhooks** flip the direction on the *server-to-server* side: instead of your
server polling a vendor's API ("any new orders yet?"), the vendor **POSTs an
HTTP request to your server** the moment an event happens.

```
Polling:                              Webhooks:
┌────────┐  HTTP GET every 10s         ┌────────┐  HTTP POST when it happens
│  you   │ ───────────────────▶        │ vendor │ ────────────────────────▶
│        │                             │        │        to your URL
│        │ ◀───── 99% empty ─────      └────────┘
└────────┘
```

### Where you already rely on webhooks
| Service | Event | What your server receives |
|---|---|---|
| Stripe | `payment_intent.succeeded` | order is paid, ship it |
| GitHub | `push`, `pull_request` | trigger CI build |
| Slack | slash command / message | respond in channel |
| Twilio | `incoming_sms` | reply to SMS |
| Shopify | `orders/create` | update inventory |

### What we'll build
1. Start a small **receiver** in this notebook (your server).
2. Register its URL with a **vendor** server (`webhook_server.py`).
3. Watch events get POSTed to us in real time.
4. Verify the HMAC signature (so we know the event really came from the vendor).
5. Simulate a flaky receiver and see the vendor **retry with backoff**.
6. Compare webhooks vs polling for a concrete workload.


## ⚙️ Step 1 — start the vendor server

In a terminal from the lab root run:

```bash
python servers/webhook_server.py   # listens on http://localhost:5005
```

It exposes `/subscriptions`, `/trigger`, `/deliveries`, `/secret`, `/health`.


In [1]:
import requests

try:
    r = requests.get("http://localhost:5005/health", timeout=2)
    print("✅ Vendor is up" if r.ok else f"⚠️ unexpected {r.status_code}")
except requests.ConnectionError:
    print("❌ Vendor not running. Start it with:")
    print("   python servers/webhook_server.py")


✅ Vendor is up


## 📬 Step 2 — run a receiver inside the notebook

A webhook receiver is just an HTTP endpoint that accepts POSTs. We'll use
FastAPI + uvicorn and run it in a background thread on port **5006** so the
vendor has somewhere to deliver to.


In [2]:
import threading, time, uvicorn
from fastapi import FastAPI, Request

received = []  # every POST the vendor makes to us lands here

receiver = FastAPI()

@receiver.post("/hook")
async def hook(request: Request):
    body = await request.body()
    received.append({
        "headers": dict(request.headers),
        "body": body.decode(),
        "at": time.time(),
    })
    return {"ok": True}

@receiver.post("/flaky")
async def flaky(request: Request):
    # Succeeds only on the 3rd attempt — used to demo retries.
    attempt = len([d for d in received if d.get("endpoint") == "flaky"]) + 1
    body = await request.body()
    received.append({
        "endpoint": "flaky",
        "attempt": attempt,
        "body": body.decode(),
        "at": time.time(),
    })
    if attempt < 3:
        # Tell the vendor we failed; it should retry.
        from fastapi.responses import JSONResponse
        return JSONResponse({"error": "temporary"}, status_code=503)
    return {"ok": True, "accepted_on_attempt": attempt}

def run_receiver():
    uvicorn.run(receiver, host="127.0.0.1", port=5006, log_level="warning")

thread = threading.Thread(target=run_receiver, daemon=True)
thread.start()

# Wait for it to start listening.
for _ in range(20):
    try:
        requests.get("http://127.0.0.1:5006/docs", timeout=0.2)
        break
    except requests.RequestException:
        time.sleep(0.2)
print("✅ Receiver listening on http://127.0.0.1:5006")


✅ Receiver listening on http://127.0.0.1:5006


## 🔗 Step 3 — register our URL with the vendor

This is the typical "add webhook" step you do in a SaaS dashboard.


In [3]:
VENDOR = "http://localhost:5005"
MY_URL = "http://127.0.0.1:5006/hook"

# Tidy up any leftover subscriptions from a previous run
for s in requests.get(f"{VENDOR}/subscriptions").json()["subscriptions"]:
    requests.delete(f"{VENDOR}/subscriptions/{s['id']}")

resp = requests.post(f"{VENDOR}/subscriptions", json={"url": MY_URL})
sub = resp.json()
print("📌 Registered:", sub)


📌 Registered: {'id': 'b432e5c7', 'url': 'http://127.0.0.1:5006/hook'}


## ⚡ Step 4 — trigger some events

Ask the vendor to send us a couple of events. Because we're the only
subscriber, both should arrive at our `/hook`.


In [4]:
received.clear()

for payload in [
    {"event": "order.created", "data": {"id": 42, "amount": 19.99}},
    {"event": "order.paid",    "data": {"id": 42, "amount": 19.99}},
]:
    r = requests.post(f"{VENDOR}/trigger", json=payload, timeout=10)
    print("→ trigger:", payload["event"], r.json())

# Give the event loop a moment to drain
time.sleep(0.3)

print(f"\n📥 Received {len(received)} webhook call(s):")
import json as _json
for rec in received:
    data = _json.loads(rec['body'])
    print(f"  • {data['event']} (delivery_id={data['delivery_id']})")


→ trigger: order.created {'delivered_to': 1}
→ trigger: order.paid {'delivered_to': 1}



📥 Received 2 webhook call(s):
  • order.created (delivery_id=a8c4ae4d93c14211b0d875d75aaf12b8)
  • order.paid (delivery_id=d136e0fc960c482f9adbfb9e9fe27c38)


## 🔐 Step 5 — verify the HMAC signature

**Never trust a webhook blindly.** Anyone on the internet can POST to your URL.
The vendor signs each body with a shared secret; we recompute the HMAC and
compare it with the `X-Webhook-Signature` header.

This is exactly what Stripe (`Stripe-Signature`) and GitHub
(`X-Hub-Signature-256`) do.


In [5]:
import hmac, hashlib

SECRET = requests.get(f"{VENDOR}/secret").json()["secret"]

def verify(body: str, header: str) -> bool:
    expected = "sha256=" + hmac.new(SECRET.encode(), body.encode(), hashlib.sha256).hexdigest()
    # hmac.compare_digest is constant-time (resists timing attacks)
    return hmac.compare_digest(expected, header)

for rec in received:
    header = rec["headers"].get("x-webhook-signature", "")
    print("verify", rec["headers"].get("x-webhook-event"), "→",
          "✅ OK" if verify(rec["body"], header) else "❌ TAMPERED")

# Demo a tampered body
tampered = received[0]["body"].replace("19.99", "0.01")
print("\nTampered body →", "✅ OK" if verify(tampered, received[0]["headers"]["x-webhook-signature"]) else "❌ TAMPERED (correctly rejected)")


verify order.created → ✅ OK
verify order.paid → ✅ OK

Tampered body → ❌ TAMPERED (correctly rejected)


## 🔁 Step 6 — retries with exponential backoff

Receivers can be flaky: a deploy, a full disk, a lock timeout. Good webhook
vendors retry failed deliveries. Let's point the vendor at our `/flaky`
endpoint, which returns **503** on the first two attempts and **200** on the
third.


In [6]:
# Swap the subscription URL to the flaky endpoint
for s in requests.get(f"{VENDOR}/subscriptions").json()["subscriptions"]:
    requests.delete(f"{VENDOR}/subscriptions/{s['id']}")

requests.post(f"{VENDOR}/subscriptions", json={"url": "http://127.0.0.1:5006/flaky"})

received.clear()
start = time.time()
requests.post(f"{VENDOR}/trigger",
              json={"event": "order.created", "data": {"id": 7}},
              timeout=20)
print(f"\n⏱️  Total vendor time: {time.time()-start:.2f}s")

# Inspect the delivery log kept by the vendor
deliveries = requests.get(f"{VENDOR}/deliveries").json()["deliveries"]
print("\n📊 Vendor-side delivery attempts:")
for d in deliveries:
    print(f"  attempt {d['attempt']:>1} → status {d.get('status_code')} ({'ok' if d['ok'] else 'FAIL'})")



⏱️  Total vendor time: 2.05s

📊 Vendor-side delivery attempts:
  attempt 1 → status 200 (ok)
  attempt 1 → status 200 (ok)
  attempt 1 → status 503 (FAIL)
  attempt 2 → status 503 (FAIL)
  attempt 3 → status 200 (ok)


## 🆚 Step 7 — webhooks vs polling: the math

Imagine you integrate with a payment provider. You expect **~100 payments an
hour**, but you want to react within a few seconds of each one.


In [7]:
payments_per_hour = 100
seconds_in_hour = 3600

# Option A: poll every 5 seconds
poll_interval = 5
polls_per_hour = seconds_in_hour / poll_interval
useful_polls = payments_per_hour          # at most 1 payment per poll
wasted_polls = polls_per_hour - useful_polls

print("Polling every 5s")
print(f"  total polls / hour : {polls_per_hour:,.0f}")
print(f"  useful polls       : {useful_polls}")
print(f"  wasted polls       : {wasted_polls:,.0f} ({wasted_polls/polls_per_hour:.0%})")
print(f"  worst-case latency : {poll_interval}s")

print("\nWebhooks")
print(f"  total requests / h : {payments_per_hour}   # 1 per event")
print(f"  wasted requests    : 0")
print(f"  worst-case latency : ~network RTT")


Polling every 5s
  total polls / hour : 720
  useful polls       : 100
  wasted polls       : 620 (86%)
  worst-case latency : 5s

Webhooks
  total requests / h : 100   # 1 per event
  wasted requests    : 0
  worst-case latency : ~network RTT


## ⚠️ Gotchas beginners hit

| Gotcha | What to do |
|---|---|
| Anyone can POST to your URL | **Always verify the signature** (Step 5). |
| Same event delivered twice (retries) | Make handlers **idempotent** — dedupe on the `delivery_id`. |
| Your receiver is slow → vendor times out & retries | **Return 200 immediately**, push the work onto a queue. |
| Events arrive out of order | Include a timestamp/sequence in the payload and ignore older ones. |
| Localhost isn't reachable from the internet | Use a tunnel (`ngrok`, `cloudflared`) or poll during dev. |
| Receiver is down during an event | Vendor retries for a bit, then **gives up** — add a periodic reconcile job as backup. |


## 🎯 When to reach for webhooks

**Great fit**
- Infrequent-but-important events: payments, signups, CI builds, deploys.
- Cross-organisation integrations (you don't control both sides).
- Anywhere you'd otherwise set up a cron job that polls for changes.

**Bad fit**
- Your users are browsers — they don't have public URLs. Use SSE/WebSockets.
- Very high event rates (>100/s per subscriber) — batch into a stream
  (Kafka, Kinesis) instead.
- You need strict ordering / exactly-once — webhooks are at-least-once.


## 🧪 Quick quiz

1. You receive the same `order.paid` webhook twice. What did the vendor most likely do, and how should your handler behave?
2. Why is `hmac.compare_digest` preferred over `==` when comparing signatures?
3. Your receiver takes 30s to process an event. The vendor times out at 10s and retries. What's the cheapest fix?


In [8]:
print("📝 Answers")
print("="*50)
print("1. It retried because an earlier attempt failed (or timed out).")
print("   Make the handler idempotent — dedupe by delivery_id.")
print("")
print("2. compare_digest is constant-time so attackers can't guess a")
print("   signature byte-by-byte using timing differences.")
print("")
print("3. Acknowledge FAST: return 200 immediately, push work to a queue")
print("   (Redis/SQS/etc) and process it asynchronously.")


📝 Answers
1. It retried because an earlier attempt failed (or timed out).
   Make the handler idempotent — dedupe by delivery_id.

2. compare_digest is constant-time so attackers can't guess a
   signature byte-by-byte using timing differences.

3. Acknowledge FAST: return 200 immediately, push work to a queue
   (Redis/SQS/etc) and process it asynchronously.


## 📚 Summary

- A **webhook** is an HTTP POST from a vendor to *your* URL when an event
  happens — the **inverse of polling**.
- You almost always want: **signature verification**, **idempotent handlers**,
  **fast ACK + async processing**, and a **reconciliation fallback**.
- Webhooks shine for low-to-medium-frequency server-to-server events. For
  browser clients you still want SSE or WebSockets (notebooks 4 and 5).
